# H&M Fashion Dataset — Exploratory Data Analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.data.loader import load_articles, load_transactions, get_image_path
from src.data.preprocess import build_item_text, filter_cold_users, temporal_split

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

## 1. Load and filter data

In [ ]:
articles = load_articles(cfg)
articles = build_item_text(articles)
transactions = load_transactions(cfg, sampled_article_ids=set(articles['article_id']))
transactions = filter_cold_users(transactions, cfg['data']['min_interactions_per_user'])
train, val, test = temporal_split(transactions, cfg['training']['val_split'], cfg['training']['test_split'])

print(f"Unique items:    {articles['article_id'].nunique():,}")
print(f"Unique users:    {transactions['customer_id'].nunique():,}")
print(f"Total transactions: {len(transactions):,}")
print(f"Train / Val / Test: {len(train):,} / {len(val):,} / {len(test):,}")
print(f"Date range: {transactions['t_dat'].min().date()} to {transactions['t_dat'].max().date()}")

## 2. Product type distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_types = articles['product_type_name'].value_counts().head(15)
axes[0].barh(top_types.index[::-1], top_types.values[::-1])
axes[0].set_title('Top 15 Product Types')
axes[0].set_xlabel('Count')

top_colours = articles['colour_group_name'].value_counts().head(15)
axes[1].barh(top_colours.index[::-1], top_colours.values[::-1])
axes[1].set_title('Top 15 Colour Groups')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. Interaction count histogram

In [ ]:
user_counts = transactions.groupby('customer_id').size()
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(user_counts.clip(upper=100), bins=50, edgecolor='black')
ax.set_title('Interactions per user (clipped at 100)')
ax.set_xlabel('Number of transactions')
ax.set_ylabel('Number of users')
plt.tight_layout()
plt.show()
print(f"Median interactions/user: {user_counts.median():.1f}")
print(f"Mean interactions/user:   {user_counts.mean():.1f}")
print(f"Max interactions/user:    {user_counts.max()}")

## 4. Sample images with captions

In [ ]:
sample = articles.sample(8, random_state=0)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img_path = get_image_path(row['article_id'], cfg)
    if img_path.exists():
        img = Image.open(img_path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'No image', ha='center', va='center', transform=ax.transAxes)
    label = f"{row['prod_name']}\n{row['colour_group_name']}\n{row['product_type_name']}"
    ax.set_title(label, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Date range and monthly transaction volume

In [ ]:
monthly = transactions.set_index('t_dat').resample('ME').size()
fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax)
ax.set_title('Monthly transaction volume')
ax.set_ylabel('Transactions')
plt.tight_layout()
plt.show()

## 6. Sample full_text field

In [ ]:
for _, row in articles.sample(3, random_state=1).iterrows():
    print(f"article_id: {row['article_id']}")
    print(f"full_text:  {row['full_text']}")
    print()